# PySWEB example: run SSEBop and SWB from a notebook with `import pysweb`

This notebook is the canonical package-backed run example. It shows how to prepare inputs and run SSEBop, then preprocess, calibrate, and run SWB directly from Python with `import pysweb`.

## Configure paths and execution toggles

Update the paths below before running the cells. All execution toggles default to `False` so the notebook is safe to open and inspect before launching downloads, preprocessing, calibration, or model runs.

In [ ]:
from pathlib import Path

RUN_PREPARE_INPUTS = False
RUN_SSEBOP = False
RUN_SWB_PREPROCESS = False
RUN_SWB_CALIBRATE = False
RUN_SWB = False

PROJECT_DIR = Path("/path/to/PySWEB")
RUN_ROOT = Path("/path/to/notebook_runs")
RUN_NAME = "example_run_name"
RUN_DATE_RANGE = "YYYY-MM-DD to YYYY-MM-DD"
CALIB_DATE_RANGE = "YYYY-MM-DD to YYYY-MM-DD"
EXTENT = [None, None, None, None]
SM_RES = 0.00025
WORKERS = 4
GEE_PROJECT = "your-gee-project"
LST_BAND = "ST_B10"
RED_BAND = "SR_B4"
NIR_BAND = "SR_B5"
NDVI_BAND = "ndvi"


def _parse_date_range(date_range):
    start_date, end_date = [part.strip() for part in date_range.split("to")]
    return start_date, end_date


def _date_tag(date_text):
    return date_text.replace("-", "")


RUN_START_DATE, RUN_END_DATE = _parse_date_range(RUN_DATE_RANGE)
RUN_START_TAG = _date_tag(RUN_START_DATE)
RUN_END_TAG = _date_tag(RUN_END_DATE)

RUN_DIR = RUN_ROOT / RUN_NAME
LANDSAT_DIR = RUN_DIR / "inputs" / "landsat"
DEM_DIR = RUN_DIR / "inputs" / "dem"
PREPARED_DEM = DEM_DIR / "nasadem.tif"
MET_RAW_DIR = RUN_DIR / "inputs" / "met" / "era5land" / "raw"
MET_STACK_DIR = RUN_DIR / "inputs" / "met" / "era5land" / "stack"
SSEBOP_OUTPUT_DIR = RUN_DIR / "outputs" / "ssebop"
SWB_PREPROCESS_DIR = RUN_DIR / "outputs" / "sweb_preprocess"
SWB_CALIBRATION_OUTPUT = RUN_DIR / "outputs" / "sweb_calibration" / "calibrated_domain_parameters.csv"
SWB_OUTPUT_DIR = RUN_DIR / "outputs" / "sweb"

ERA5LAND_PRECIP_SOURCE = MET_STACK_DIR / f"precipitation_daily_{RUN_START_DATE}_{RUN_END_DATE}.nc"
SSEBOP_ET_SOURCE = SSEBOP_OUTPUT_DIR / f"et_daily_ssebop_{RUN_START_DATE}_{RUN_END_DATE}.nc"
PREPARED_RAIN_SOURCE = ERA5LAND_PRECIP_SOURCE
PREPARED_ET_SOURCE = SSEBOP_ET_SOURCE

for path in [
    LANDSAT_DIR,
    DEM_DIR,
    MET_RAW_DIR,
    MET_STACK_DIR,
    SSEBOP_OUTPUT_DIR,
    SWB_PREPROCESS_DIR,
    SWB_CALIBRATION_OUTPUT.parent,
    SWB_OUTPUT_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

print(f"Notebook run directory: {RUN_DIR}")
print(f"Run date range: {RUN_DATE_RANGE}")
if RUN_SWB_CALIBRATE:
    print(f"Calibration date range: {CALIB_DATE_RANGE}")
print(f"Prepared rain source: {PREPARED_RAIN_SOURCE}")
print(f"Prepared ET source: {PREPARED_ET_SOURCE}")
print(f"Calibration CSV target: {SWB_CALIBRATION_OUTPUT}")
print(f"GEE project: {GEE_PROJECT}")
print(f"Prepared DEM target: {PREPARED_DEM}")


In [ ]:
import sys

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import pysweb

if RUN_PREPARE_INPUTS:
    pysweb.ssebop.prepare_inputs(
        date_range = RUN_DATE_RANGE,
        extent = EXTENT,
        met_source = "era5land",
        gee_project = GEE_PROJECT,
        landsat_dir = str(LANDSAT_DIR),
        met_raw_dir = str(MET_RAW_DIR),
        met_stack_dir = str(MET_STACK_DIR),
        dem_dir = str(DEM_DIR),
    )
    if RUN_SWB_CALIBRATE:
        calib_start_date, calib_end_date = _parse_date_range(CALIB_DATE_RANGE)
        if (calib_start_date, calib_end_date) != (RUN_START_DATE, RUN_END_DATE):
            pysweb.ssebop.prepare_inputs(
                date_range = CALIB_DATE_RANGE,
                extent = EXTENT,
                met_source = "era5land",
                gee_project = GEE_PROJECT,
                landsat_dir = str(LANDSAT_DIR),
                met_raw_dir = str(MET_RAW_DIR),
                met_stack_dir = str(MET_STACK_DIR),
                dem_dir = str(DEM_DIR),
            )
else:
    print("Set RUN_PREPARE_INPUTS = True to download Landsat, NASADEM, and ERA5-Land inputs.")


In [ ]:
if RUN_SSEBOP:
    pysweb.ssebop.run(
        date_range = RUN_DATE_RANGE,
        landsat_dir = str(LANDSAT_DIR),
        met_dir = str(MET_STACK_DIR),
        dem = str(PREPARED_DEM),
        output_dir = str(SSEBOP_OUTPUT_DIR),
        lst_band = LST_BAND,
        red_band = RED_BAND,
        nir_band = NIR_BAND,
        ndvi_band = NDVI_BAND,
        workers = WORKERS,
    )
    if RUN_SWB_CALIBRATE:
        calib_start_date, calib_end_date = _parse_date_range(CALIB_DATE_RANGE)
        if (calib_start_date, calib_end_date) != (RUN_START_DATE, RUN_END_DATE):
            pysweb.ssebop.run(
                date_range = CALIB_DATE_RANGE,
                landsat_dir = str(LANDSAT_DIR),
                met_dir = str(MET_STACK_DIR),
                dem = str(PREPARED_DEM),
                output_dir = str(SSEBOP_OUTPUT_DIR),
                lst_band = LST_BAND,
                red_band = RED_BAND,
                nir_band = NIR_BAND,
                ndvi_band = NDVI_BAND,
                workers = WORKERS,
            )
else:
    print("Set RUN_SSEBOP = True to execute the package-backed SSEBop workflow.")


In [ ]:
import matplotlib.pyplot as plt
import xarray as xr
from IPython.display import display

ssebop_nc = SSEBOP_OUTPUT_DIR / f"et_daily_ssebop_{RUN_START_DATE}_{RUN_END_DATE}.nc"

if ssebop_nc.exists():
    ds_ssebop = xr.open_dataset(ssebop_nc)
    display(ds_ssebop)
    ds_ssebop["ET"].isel(time=0).plot(cmap="viridis")
    plt.title(f"SSEBop ET on {str(ds_ssebop.time.values[0])[:10]}")
    plt.show()
else:
    print(f"Run SSEBop first or update the path: {ssebop_nc}")


## Package-backed SWB preprocess, calibrate, and run

The cells below keep execution disabled by default and use neutral placeholders for prepared rain and ET sources. Replace `PREPARED_RAIN_SOURCE`, `PREPARED_ET_SOURCE`, and other run-specific paths before enabling the SWB cells.

`SM_RES` is the SWB preprocess target grid resolution.


In [ ]:
import pandas as pd
import xarray as xr


def _clip_window_to_et_coverage(start_date, end_date, et_source, label):
    swb_start_date = start_date
    swb_end_date = end_date
    if et_source.exists():
        with xr.open_dataset(et_source) as ds_et:
            if "time" in ds_et.coords and ds_et.sizes.get("time", 0) > 0:
                et_dates = pd.to_datetime(ds_et.time.values).normalize()
                swb_start_date = max(pd.Timestamp(start_date), et_dates.min()).strftime("%Y-%m-%d")
                swb_end_date = min(pd.Timestamp(end_date), et_dates.max()).strftime("%Y-%m-%d")
                print(f"{label} SWB date range clipped to ET coverage: {swb_start_date} to {swb_end_date}")
    return swb_start_date, swb_end_date


def _preprocess_swb_window(label, start_date, end_date, rain_source, et_source, include_reference_ssm):
    print(f"Preprocessing {label} SWB window: {start_date} to {end_date}")
    pysweb.swb.preprocess(
        start_date = start_date,
        end_date = end_date,
        extent = EXTENT,
        gee_project = GEE_PROJECT,
        sm_res = SM_RES,
        rain_file = str(rain_source),
        rain_var = "precipitation",
        et_file = str(et_source),
        e_var = "E",
        et_var = "ET",
        t_var = "T",
        output_dir = str(SWB_PREPROCESS_DIR),
        soil_source = "openlandmap",
        openlandmap_missing_soc_g_per_kg = 5.0,
        skip_reference_ssm = not include_reference_ssm,
        workers = WORKERS,
    )


RUN_SWB_START_DATE, RUN_SWB_END_DATE = _clip_window_to_et_coverage(
    RUN_START_DATE,
    RUN_END_DATE,
    PREPARED_ET_SOURCE,
    "Run",
)
RUN_SWB_START_TAG = _date_tag(RUN_SWB_START_DATE)
RUN_SWB_END_TAG = _date_tag(RUN_SWB_END_DATE)
CALIB_SAME_AS_RUN = False

if RUN_SWB_CALIBRATE:
    CALIB_START_DATE, CALIB_END_DATE = _parse_date_range(CALIB_DATE_RANGE)
    CALIB_RAIN_SOURCE = MET_STACK_DIR / f"precipitation_daily_{CALIB_START_DATE}_{CALIB_END_DATE}.nc"
    CALIB_ET_SOURCE = SSEBOP_OUTPUT_DIR / f"et_daily_ssebop_{CALIB_START_DATE}_{CALIB_END_DATE}.nc"
    CALIB_SWB_START_DATE, CALIB_SWB_END_DATE = _clip_window_to_et_coverage(
        CALIB_START_DATE,
        CALIB_END_DATE,
        CALIB_ET_SOURCE,
        "Calibration",
    )
    CALIB_SWB_START_TAG = _date_tag(CALIB_SWB_START_DATE)
    CALIB_SWB_END_TAG = _date_tag(CALIB_SWB_END_DATE)
    CALIB_SAME_AS_RUN = (
        CALIB_SWB_START_DATE == RUN_SWB_START_DATE
        and CALIB_SWB_END_DATE == RUN_SWB_END_DATE
    )

if RUN_SWB_PREPROCESS:
    _preprocess_swb_window(
        "run",
        RUN_SWB_START_DATE,
        RUN_SWB_END_DATE,
        PREPARED_RAIN_SOURCE,
        PREPARED_ET_SOURCE,
        include_reference_ssm = RUN_SWB_CALIBRATE and CALIB_SAME_AS_RUN,
    )
    if RUN_SWB_CALIBRATE and not CALIB_SAME_AS_RUN:
        _preprocess_swb_window(
            "calibration",
            CALIB_SWB_START_DATE,
            CALIB_SWB_END_DATE,
            CALIB_RAIN_SOURCE,
            CALIB_ET_SOURCE,
            include_reference_ssm = True,
        )
else:
    print("Set RUN_SWB_PREPROCESS = True after pointing prepared rain and ET sources to your prepared inputs.")


In [ ]:
RUN_RAIN_NC = SWB_PREPROCESS_DIR / f"rain_daily_{RUN_SWB_START_TAG}_{RUN_SWB_END_TAG}.nc"
RUN_EFFECTIVE_PRECIP_NC = SWB_PREPROCESS_DIR / f"effective_precip_daily_{RUN_SWB_START_TAG}_{RUN_SWB_END_TAG}.nc"
RUN_ET_NC = SWB_PREPROCESS_DIR / f"et_daily_{RUN_SWB_START_TAG}_{RUN_SWB_END_TAG}.nc"
RUN_T_NC = SWB_PREPROCESS_DIR / f"t_daily_{RUN_SWB_START_TAG}_{RUN_SWB_END_TAG}.nc"

if RUN_SWB_CALIBRATE:
    if CALIB_SAME_AS_RUN:
        CALIB_EFFECTIVE_PRECIP_NC = RUN_EFFECTIVE_PRECIP_NC
        CALIB_ET_NC = RUN_ET_NC
        CALIB_T_NC = RUN_T_NC
    else:
        CALIB_EFFECTIVE_PRECIP_NC = SWB_PREPROCESS_DIR / f"effective_precip_daily_{CALIB_SWB_START_TAG}_{CALIB_SWB_END_TAG}.nc"
        CALIB_ET_NC = SWB_PREPROCESS_DIR / f"et_daily_{CALIB_SWB_START_TAG}_{CALIB_SWB_END_TAG}.nc"
        CALIB_T_NC = SWB_PREPROCESS_DIR / f"t_daily_{CALIB_SWB_START_TAG}_{CALIB_SWB_END_TAG}.nc"
    CALIB_REFERENCE_SSM_NC = SWB_PREPROCESS_DIR / f"reference_ssm_daily_{CALIB_SWB_START_TAG}_{CALIB_SWB_END_TAG}.nc"

    pysweb.swb.calibrate(
        effective_precip = str(CALIB_EFFECTIVE_PRECIP_NC),
        et = str(CALIB_ET_NC),
        t = str(CALIB_T_NC),
        soil_dir = str(SWB_PREPROCESS_DIR),
        reference_ssm = str(CALIB_REFERENCE_SSM_NC),
        date_range = [CALIB_SWB_START_DATE, CALIB_SWB_END_DATE],
        output = str(SWB_CALIBRATION_OUTPUT),
        max_iter = 10,
        workers = WORKERS,
    )
else:
    print("Set RUN_SWB_CALIBRATE = True after SWB preprocess outputs are ready.")
    print(f"Calibration output will be written to: {SWB_CALIBRATION_OUTPUT}")


In [ ]:
import pandas as pd

run_kwargs = {
    "precip": str(RUN_RAIN_NC),
    "effective_precip": str(RUN_EFFECTIVE_PRECIP_NC),
    "et": str(RUN_ET_NC),
    "t": str(RUN_T_NC),
    "soil_dir": str(SWB_PREPROCESS_DIR),
    "output_dir": str(SWB_OUTPUT_DIR),
    "start_date": RUN_SWB_START_DATE,
    "end_date": RUN_SWB_END_DATE,
    "workers": WORKERS,
}

if RUN_SWB_CALIBRATE and SWB_CALIBRATION_OUTPUT.exists():
    calibrated = pd.read_csv(SWB_CALIBRATION_OUTPUT).iloc[0]
    run_kwargs["diff_factor"] = float(calibrated.get("diff_factor", 1e3))
    run_kwargs["sm_max_factor"] = float(calibrated.get("sm_max_factor", 1.0))
    run_kwargs["sm_min_factor"] = float(calibrated.get("sm_min_factor", 1.0))
    run_kwargs["root_beta"] = float(calibrated.get("root_beta", 0.961))
elif SWB_CALIBRATION_OUTPUT.exists():
    print(f"RUN_SWB_CALIBRATE is False; using default SWB parameters and ignoring calibration CSV: {SWB_CALIBRATION_OUTPUT}")
else:
    print(f"Calibration CSV not found yet, using default SWB parameters: {SWB_CALIBRATION_OUTPUT}")

if RUN_SWB:
    pysweb.swb.run(**run_kwargs)
else:
    print("Set RUN_SWB = True after SWB preprocess outputs are ready and optional calibration parameters are available.")
    print(f"Expected SWB preprocess directory: {SWB_PREPROCESS_DIR}")


In [ ]:
swb_outputs = sorted(SWB_OUTPUT_DIR.glob("SWEB_RZSM*.nc"))

if swb_outputs:
    ds_swb = xr.open_dataset(swb_outputs[-1])
    display(ds_swb)
    preview_var = "profile_sm" if "profile_sm" in ds_swb.data_vars else list(ds_swb.data_vars)[0]
    ds_swb[preview_var].isel(time=0).plot(cmap="Blues")
    plt.title(f"{preview_var} on {str(ds_swb.time.values[0])[:10]}")
    plt.show()
else:
    print(f"No SWB output found in {SWB_OUTPUT_DIR}")